# Long-form input data

Model data often arrives in **long form** (one row per key combination): a CSV
export, a database query or a polars pipeline. Each row holds a few key
columns, such as generator and snapshot, and one value column. linopy works on
dimension-labelled arrays instead. This tutorial shows the two tools that
bridge the gap:

- `linopy.from_long(frame, coords, default)` pivots a long-form frame
  into a `DataArray`. Use it wherever linopy accepts a constant: bounds,
  coefficients, right-hand sides and penalties.
- `linopy.Relation(frame, member, group)` maps the members of a dimension to
  groups, possibly several groups per member. `expr.groupby(relation).sum()`
  adds each member to each of its groups.

Both accept pandas and polars frames. We build a small three-country dispatch
model to show them in action.

> **Note:** both tools are new and their API may still change in minor
> releases. They emit a `linopy.EvolvingAPIWarning` once per session.

## Coordinates and raw data

The coordinates define the model dimensions. The input data are plain frames,
as you would read them from files.

In [ ]:
import pandas as pd
import polars as pl

import linopy

snapshots = pd.date_range("2030-01-01", periods=4, freq="6h", name="snapshot")
gens = pd.Index(["coal_de", "wind_de", "solar_fr", "nuke_fr", "gas_nl"], name="gen")
buses = pd.Index(["DE", "FR", "NL"], name="bus")
links = pd.Index(["DE-FR", "DE-NL", "FR-NL"], name="link")

generators = pd.DataFrame(
    {
        "gen": gens,
        "bus": ["DE", "DE", "FR", "FR", "NL"],
        "p_nom": [300.0, 200, 150, 400, 250],
        "cost": [30.0, 0, 0, 10, 50],
        "co2": [0.9, 0, 0, 0, 0.4],
    }
)
generators

Availability is time-dependent, but only for the renewable generators.
The frame lists those rows only. Here it is a polars frame with datetime keys.

In [ ]:
availability = pl.DataFrame(
    {
        "gen": ["wind_de"] * 4 + ["solar_fr"] * 4,
        "snapshot": list(snapshots) * 2,
        "p_max_pu": [0.8, 0.6, 0.3, 0.5, 0.0, 0.7, 0.9, 0.1],
    }
)
availability

## Pivoting with `from_long`

`from_long` treats every column named after a dimension of `coords` as a key
column. Exactly one other column must remain: the value column, which also
names the result. The result spans the key dimensions with their full
coordinates. Key combinations missing from the frame take `default`, which is
NaN unless you set it.

The availability frame has no rows for the conventional generators. With
`default=1.0` they are fully available:

In [ ]:
p_max_pu = linopy.from_long(availability, coords=[gens, snapshots], default=1.0)
p_max_pu

A frame that lacks a key column for a dimension of `coords` gives a result
without that dimension. Here `p_nom` spans only `gen`. linopy broadcasts it
over `snapshot` where you use it, just like any other `DataArray`.

A wide table has several value columns, so select one of them together with
its keys.

In [ ]:
p_nom = linopy.from_long(generators[["gen", "p_nom"]], coords=[gens])
p_nom

The pivoted arrays plug straight into the model, here as the upper bound
of the dispatch variable `p`.

In [ ]:
m = linopy.Model()

p = m.add_variables(lower=0, upper=p_nom * p_max_pu, coords=[gens, snapshots], name="p")
f = m.add_variables(lower=-200, upper=200, coords=[links, snapshots], name="f")
p.upper

Demand comes as another long-form frame over `bus` and `snapshot`. The key
columns may come in any order; the result follows the column order of the
frame.

In [ ]:
demand = pl.DataFrame(
    {
        "bus": ["DE"] * 4 + ["FR"] * 4 + ["NL"] * 4,
        "snapshot": list(snapshots) * 3,
        "load": [400.0, 450, 500, 420, 300, 350, 380, 320, 150, 200, 220, 180],
    }
)
load = linopy.from_long(demand, coords=[buses, snapshots])

## Grouping with `Relation`

A `Relation` reads member-group pairs from a frame of exactly two columns,
the member and the group column. Grouping by
it sums each member into its group. The generator table already relates every
generator to its bus, so we select these two columns:

In [ ]:
supply = p.groupby(linopy.Relation(generators[["gen", "bus"]], "gen", "bus")).sum()
supply

Links connect two buses. A link withdraws its flow at `bus0` and injects
it at `bus1`. One relation per port collects the flows at each bus.

The result covers only the groups that appear in the frame: no link starts at
`NL`. Reindex onto all buses and fill the gaps with zero before you combine
the terms.

In [ ]:
ports = pd.DataFrame(
    {"link": links, "bus0": ["DE", "DE", "FR"], "bus1": ["FR", "NL", "NL"]}
)
withdrawal = f.groupby(linopy.Relation(ports[["link", "bus0"]], "link", "bus0")).sum()
injection = f.groupby(linopy.Relation(ports[["link", "bus1"]], "link", "bus1")).sum()

withdrawal = withdrawal.rename(bus0="bus").reindex(bus=buses).fillna(0)
injection = injection.rename(bus1="bus").reindex(bus=buses).fillna(0)

m.add_constraints(supply + injection - withdrawal == load, name="balance")

### Many-to-many relations

A member may appear in several rows. Each row adds the member to one more
group. This fits overlapping policy zones: the coal plant counts towards the
German and the EU emission cap, the gas plant towards the Dutch and the EU
cap. Generators without emissions are not listed, so they drop out.

In [ ]:
zones = pd.DataFrame(
    {
        "gen": ["coal_de", "coal_de", "gas_nl", "gas_nl"],
        "zone": ["DE", "EU", "NL", "EU"],
    }
)
co2 = linopy.from_long(generators[["gen", "co2"]], coords=[gens])
emissions = (
    (co2 * p).sum("snapshot").groupby(linopy.Relation(zones, "gen", "zone")).sum()
)
emissions

The groups come out sorted. Pivot the caps onto the same zone
coordinates, so the right-hand side lines up with the expression.

In [ ]:
caps = pd.DataFrame({"zone": ["DE", "NL", "EU"], "cap": [800.0, 500, 1100]})
cap = linopy.from_long(caps, coords=[emissions.indexes["zone"]])
m.add_constraints(emissions <= cap, name="co2_cap")

## Solving and getting long-form results back

The objective uses one more pivoted column. Instead of listing the
coordinates, pass those of the variable the result is used with,
`coords=p.coords`. After solving, `to_dataframe()`
followed by `reset_index()` turns any solution back into long form.

In [ ]:
cost = linopy.from_long(generators[["gen", "cost"]], coords=p.coords)
m.add_objective((cost * p).sum())
m.solve(solver_name="highs", output_flag=False)

In [ ]:
p.solution.to_dataframe().reset_index().head()

In [ ]:
m.constraints["co2_cap"].dual

## Validation

`from_long` and `Relation` check the input and fail fast instead of
silently dropping rows. Among others, they raise for:

- a column that is neither a key column nor the one value column, e.g. a
  misspelled key column,
- duplicate key combinations,
- keys outside the coordinates, e.g. a typo in a generator name,
- null keys or null values,
- key dtypes that differ from the coordinates, e.g. string keys for an
  integer dimension.

In [ ]:
typo = pd.DataFrame({"gen": ["wind_de", "wind_uk"], "p_nom": [1.0, 2.0]})
try:
    linopy.from_long(typo, coords=[gens])
except ValueError as e:
    print(e)

In [ ]:
duplicate = pd.DataFrame({"gen": ["wind_de", "wind_de"], "p_nom": [1.0, 2.0]})
try:
    linopy.from_long(duplicate, coords=[gens])
except ValueError as e:
    print(e)

A bare frame is not a valid grouper, because linopy cannot tell the member
column from the group column. Wrap it in a `Relation`:

In [ ]:
try:
    p.groupby(zones).sum()
except TypeError as e:
    print(e)

## Notes

- A pandas frame needs an unnamed index. Call `reset_index()` to turn index
  levels into key columns.
- `from_long` returns the full array over the key dimensions. A sparsely
  filled frame still costs the memory of the full array.
- In a sparse model, `Model(sparse=True)`, grouping by a `Relation` keeps the
  expression sparse. See the sparse models tutorial.
- Grouping a quadratic expression by a `Relation` is not supported.